# Experimentación, Fine-Tuning y Registro en MLflow (`experimentos_proyecto.ipynb`)

## 1. Diseño Experimental y Espacio de Búsqueda

* **Presupuesto de experimentación (101 corridas registradas):**
  * **3 Líneas Base:** Baseline_Dummy_Mayoritaria, Baseline_Regla_Negocio_14d y Baseline_Regla_1_Sesion
  * **4 Candidatos por defecto:** `LogisticRegression`, `RandomForest`, `HistGradientBoosting` y `MLPClassifier`.
  * **Fine-Tuning (`RandomizedSearchCV`, `30` configuraciones × `3` mejores familias = `90` *nested runs*):** Se elige búsqueda aleatoria reproducible (`random_state=42`) sobre `TimeSeriesSplit(n_splits=5)` porque explora eficientemente hiperparámetros continuos en escala logarítmica con presupuesto fijo de `30` evaluaciones por familia:
    1. **`HistGradientBoostingClassifier` (Boosting):** `learning_rate` en escala log-uniforme `[0.01, 0.3]`, `l2_regularization` en escala log-uniforme `[1e-2, 20.0]`, `max_depth` en `[3, 4, 5, 6, None]`, `max_leaf_nodes` en `[15, 31, 63]` y `min_samples_leaf` en `[10, 20, 30, 50]`.
    2. **`MLPClassifier` (Red Neuronal):** `alpha` (penalización L2) en escala log-uniforme `[1e-4, 5.0]`, `learning_rate_init` en escala log-uniforme `[1e-4, 1e-2]` y `hidden_layer_sizes` en `[(32,), (64,), (100,), (64, 32), (128, 64)]`.
    3. **`LogisticRegression` (Lineal Regularizado):** `C` (inverso de fuerza de regularización) en escala log-uniforme `[1e-3, 100.0]` y `class_weight` en `[None, "balanced"]`.
  * **1 Corrida Final (`candidato = final`):** Entrenamiento y registro del ganador en el Model Registry como `oralsolutions-modelo` con alias `@champion` (y el segundo mejor como `@challenger`), seguido de su recarga y verificación exacta.

In [1]:
import sys; from pathlib import Path
sys.path.insert(0, str(Path('..').resolve()))
import subprocess
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import loguniform

import mlflow
import mlflow.sklearn
from mlflow.models import infer_signature
from mlflow.tracking import MlflowClient

from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import TimeSeriesSplit, cross_validate, RandomizedSearchCV
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    f1_score,
    recall_score,
    precision_score,
)

from src.pipeline import (
    obtener_datos_divididos,
    construir_pipeline,
    obtener_hash_datos,
    FECHA_CORTE,
    SEMILLA,
)

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

TRACKING_URI = "http://127.0.0.1:5000"
EXPERIMENT_NAME = "OralSolutions"
MODEL_REGISTRY_NAME = "oralsolutions-modelo"
UMBRAL_NEGOCIO = 0.60

mlflow.set_tracking_uri(TRACKING_URI)
client = MlflowClient(tracking_uri=TRACKING_URI)

# Obtener commit actual de Git para trazabilidad
try:
    GIT_COMMIT = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip()
except Exception:
    GIT_COMMIT = "sin-git"

HASH_DATOS = obtener_hash_datos()
X_train, X_test, y_train, y_test = obtener_datos_divididos()
cv_temporal = TimeSeriesSplit(n_splits=5)

# Crear o limpiar experimento para que corridas_proyecto.csv salga de una corrida limpia
exp = client.get_experiment_by_name(EXPERIMENT_NAME)
if exp is None:
    exp_id = client.create_experiment(EXPERIMENT_NAME)
else:
    exp_id = exp.experiment_id
    corridas_viejas = client.search_runs(experiment_ids=[exp_id])
    for r in corridas_viejas:
        client.delete_run(r.info.run_id)

mlflow.set_experiment(EXPERIMENT_NAME)
print(f"Conectado a MLflow: {TRACKING_URI} | Experimento: {EXPERIMENT_NAME} (ID={exp_id})")
print(f"SHA-256 dataset   : {HASH_DATOS[:16]}...{HASH_DATOS[-8:]} | Git commit: {GIT_COMMIT}")


class ReglaNegocio14Dias(BaseEstimator, ClassifierMixin):
    def __init__(self, max_dias_ritmo: float = 14.0, min_friccion: int = 1):
        self.max_dias_ritmo = max_dias_ritmo
        self.min_friccion = min_friccion

    def fit(self, X: pd.DataFrame, y=None):
        self.classes_ = np.array([0, 1])
        return self

    def predict_proba(self, X: pd.DataFrame) -> np.ndarray:
        alerta = (
            (X["ritmo_dias_por_sesion"] > self.max_dias_ritmo)
            | (X["friccion_previa"] >= self.min_friccion)
        ).astype(float)
        p1 = np.where(alerta == 1.0, 0.80, 0.20)
        return np.column_stack([1.0 - p1, p1])

    def predict(self, X: pd.DataFrame) -> np.ndarray:
        return (self.predict_proba(X)[:, 1] >= UMBRAL_NEGOCIO).astype(int)

class Regla1SesionRestante(BaseEstimator, ClassifierMixin):
    def fit(self, X: pd.DataFrame, y=None):
        self.classes_ = np.array([0, 1])
        return self

    def predict_proba(self, X: pd.DataFrame) -> np.ndarray:
        alerta = ((X["sesiones_planificadas"] - X["numero_sesion"]) == 1).astype(float)
        p1 = np.where(alerta == 1.0, 0.99, 0.13)
        return np.column_stack([1.0 - p1, p1])

    def predict(self, X: pd.DataFrame) -> np.ndarray:
        return (self.predict_proba(X)[:, 1] >= UMBRAL_NEGOCIO).astype(int)

def registrar_corrida_simple(
    run_name: str,
    fase: str,
    familia: str,
    estimador,
    params_extra: dict | None = None,
    candidato_tag: str = "no",
) -> dict:
    with mlflow.start_run(run_name=run_name) as run:
        t0 = time.perf_counter()
        cv_res = cross_validate(
            estimador,
            X_train,
            y_train,
            cv=cv_temporal,
            scoring={"pr_auc": "average_precision", "roc_auc": "roc_auc"},
            n_jobs=-1,
        )
        estimador.fit(X_train, y_train)
        segundos = time.perf_counter() - t0

        prob_train = estimador.predict_proba(X_train)[:, 1]
        prob_test = estimador.predict_proba(X_test)[:, 1]
        pred_test = (prob_test >= UMBRAL_NEGOCIO).astype(int)

        pr_train = float(average_precision_score(y_train, prob_train))
        pr_cv_mean = float(np.nanmean(cv_res["test_pr_auc"]))
        pr_cv_std = float(np.nanstd(cv_res["test_pr_auc"]))
        pr_test = float(average_precision_score(y_test, prob_test))

        # Parámetros de trazabilidad y del modelo
        mlflow.log_params({
            "tabla_origen": "obt_riesgo_abandono.csv",
            "fecha_corte": FECHA_CORTE,
            "semilla": SEMILLA,
            "hash_dataset": HASH_DATOS,
            "umbral_negocio": UMBRAL_NEGOCIO,
            **(params_extra or {}),
        })

        # Métricas
        mlflow.log_metrics({
            "pr_auc_train": round(pr_train, 5),
            "pr_auc_cv_mean": round(pr_cv_mean, 5) if not np.isnan(pr_cv_mean) else -1.0,
            "pr_auc_cv_std": round(pr_cv_std, 5) if not np.isnan(pr_cv_std) else -1.0,
            "pr_auc_test": round(pr_test, 5),
            "brecha_train_test": round(pr_train - pr_test, 5),
            "roc_auc_test": round(float(roc_auc_score(y_test, prob_test)), 5),
            "f1_test_060": round(float(f1_score(y_test, pred_test, zero_division=0)), 5),
            "recall_test_060": round(float(recall_score(y_test, pred_test, zero_division=0)), 5),
            "precision_test_060": round(float(precision_score(y_test, pred_test, zero_division=0)), 5),
            "segundos_entrenamiento": round(segundos, 4),
        })

        # Tags
        mlflow.set_tags({
            "fase": fase,
            "familia": familia,
            "candidato": candidato_tag,
            "git_commit": GIT_COMMIT,
        })

        # Artefacto del pipeline entrenado con firma y ejemplo de entrada
        input_example = X_train.iloc[:5].copy()
        signature = infer_signature(input_example, estimador.predict_proba(input_example))
        mlflow.sklearn.log_model(
            sk_model=estimador,
            artifact_path="model",
            signature=signature,
            input_example=input_example,
        )

        return {
            "run_id": run.info.run_id,
            "run_name": run_name,
            "fase": fase,
            "familia": familia,
            "pr_auc_cv_mean": pr_cv_mean,
            "pr_auc_cv_std": pr_cv_std,
            "pr_auc_test": pr_test,
        }


# 1. Registrar las 2 Líneas Base y los 4 Candidatos por defecto
corridas_base_cand = [
    ("Baseline_Dummy_Mayoritaria", "baseline", "Linea_Base",
    construir_pipeline(DummyClassifier(strategy="most_frequent")), {"strategy": "most_frequent"}),
    ("Baseline_Regla_Negocio_14d", "baseline", "Linea_Base",
    ReglaNegocio14Dias(), {"max_dias_ritmo": 14.0, "min_friccion": 1}),
    ("Baseline_Regla_1_Sesion", "baseline", "Linea_Base",
    Regla1SesionRestante(), {"regla": "sesiones_restantes == 1"}),
    ("LogisticRegression_default", "candidato", "Lineal_Regularizado",
    construir_pipeline(LogisticRegression(random_state=SEMILLA), escalar_numericas=True), {"C": 1.0}),
    ("RandomForest_default", "candidato", "Bagging",
    construir_pipeline(RandomForestClassifier(random_state=SEMILLA), escalar_numericas=False), {"n_estimators": 100}),
    ("HistGradientBoosting_default", "candidato", "Boosting",
    construir_pipeline(HistGradientBoostingClassifier(random_state=SEMILLA), escalar_numericas=False), {"learning_rate": 0.1}),
    ("MLPClassifier_default", "candidato", "Red_Neuronal",
    construir_pipeline(MLPClassifier(random_state=SEMILLA, max_iter=500), escalar_numericas=True), {"alpha": 0.0001}),
]

resumen_inicial = [registrar_corrida_simple(*c) for c in corridas_base_cand]
display(pd.DataFrame(resumen_inicial))

Conectado a MLflow: http://127.0.0.1:5000 | Experimento: OralSolutions (ID=1)
SHA-256 dataset   : 937af6f343e5de34...a767cf60 | Git commit: 295f38d


🏃 View run Baseline_Dummy_Mayoritaria at: http://127.0.0.1:5000/#/experiments/1/runs/3ec65740da8949e6aa708f2e9d0febb4
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


🏃 View run Baseline_Regla_Negocio_14d at: http://127.0.0.1:5000/#/experiments/1/runs/15bea8e818634529a0545377b9e2a165
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


🏃 View run Baseline_Regla_1_Sesion at: http://127.0.0.1:5000/#/experiments/1/runs/ad372d1a51a646f687abd0d5aaeacf8f
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


🏃 View run LogisticRegression_default at: http://127.0.0.1:5000/#/experiments/1/runs/f7f3ce582dd2435a814ed2d5e3b23a32
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


🏃 View run RandomForest_default at: http://127.0.0.1:5000/#/experiments/1/runs/4adf46dd75f04013bc5bac01ca2b401d
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


🏃 View run HistGradientBoosting_default at: http://127.0.0.1:5000/#/experiments/1/runs/69f3f6c3134d4f3cab4765972ce37597
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


🏃 View run MLPClassifier_default at: http://127.0.0.1:5000/#/experiments/1/runs/d048f43e7aa34e04a58040b9bf514694
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


,run_id,run_name,fase,familia,pr_auc_cv_mean,pr_auc_cv_std,pr_auc_test
0,3ec65740da8949e6aa708f2e9d0febb4,Baseline_Dummy_Mayoritaria,baseline,Linea_Base,0.625581,0.024830,0.623487
1,15bea8e818634529a0545377b9e2a165,Baseline_Regla_Negocio_14d,baseline,Linea_Base,NaN,NaN,0.655145
2,ad372d1a51a646f687abd0d5aaeacf8f,Baseline_Regla_1_Sesion,baseline,Linea_Base,NaN,NaN,0.964176
3,f7f3ce582dd2435a814ed2d5e3b23a32,LogisticRegression_default,candidato,Lineal_Regularizado,0.981700,0.004369,0.981596
4,4adf46dd75f04013bc5bac01ca2b401d,RandomForest_default,candidato,Bagging,0.979683,0.004108,0.981448
5,69f3f6c3134d4f3cab4765972ce37597,HistGradientBoosting_default,candidato,Boosting,0.980565,0.005361,0.984271
6,d048f43e7aa34e04a58040b9bf514694,MLPClassifier_default,candidato,Red_Neuronal,0.982556,0.004233,0.983315


Se registró en el experimento `OralSolutions` de MLflow la huella SHA-256 del dataset limpio, la fecha de corte (`2025-08-01`) y las **7 corridas base** (3 líneas base y 4 modelos candidatos por defecto). Se confirma que los modelos de ML superan ampliamente a las referencias débiles (`0.6235` del Dummy y `0.6551` de la regla de 14 días) y mejoran también a la regla fuerte del EDA (`0.9642` en `Baseline_Regla_1_Sesion`) al capturar riesgo temprano cuando aún faltan 2 o 3 sesiones. Destacan en validación cruzada `MLPClassifier` (`0.98256 ± 0.00423`), `LogisticRegression` (`0.98170 ± 0.00437`) y `HistGradientBoosting` (`0.98057 ± 0.00536`), que pasan a la fase de ajuste fino.

In [2]:
N_CONFIGS = 30

busquedas_tuning = [
    (
        "Tuning_HistGradientBoosting",
        "Boosting",
        construir_pipeline(HistGradientBoostingClassifier(random_state=SEMILLA), escalar_numericas=False),
        {
            "modelo__learning_rate": loguniform(0.01, 0.3),
            "modelo__l2_regularization": loguniform(1e-2, 20.0),
            "modelo__max_depth": [3, 4, 5, 6, None],
            "modelo__max_leaf_nodes": [15, 31, 63],
            "modelo__min_samples_leaf": [10, 20, 30, 50],
        },
    ),
    (
        "Tuning_MLPClassifier",
        "Red_Neuronal",
        construir_pipeline(MLPClassifier(max_iter=500, random_state=SEMILLA), escalar_numericas=True),
        {
            "modelo__alpha": loguniform(1e-4, 5.0),
            "modelo__learning_rate_init": loguniform(1e-4, 1e-2),
            "modelo__hidden_layer_sizes": [(32,), (64,), (100,), (64, 32), (128, 64)],
        },
    ),
    (
        "Tuning_LogisticRegression",
        "Lineal_Regularizado",
        construir_pipeline(LogisticRegression(max_iter=1000, random_state=SEMILLA), escalar_numericas=True),
        {
            "modelo__C": loguniform(1e-3, 100.0),
            "modelo__class_weight": [None, "balanced"],
        },
    ),
]

mejores_tuneados = []

for nombre_padre, familia, pipe_base, espacio_params in busquedas_tuning:
    with mlflow.start_run(run_name=nombre_padre) as parent_run:
        t0_padre = time.perf_counter()
        search = RandomizedSearchCV(
            estimator=pipe_base,
            param_distributions=espacio_params,
            n_iter=N_CONFIGS,
            scoring="average_precision",
            cv=cv_temporal,
            random_state=SEMILLA,
            n_jobs=-1,
            return_train_score=True,
        )
        search.fit(X_train, y_train)
        seg_padre = time.perf_counter() - t0_padre

        # Registrar cada una de las 30 configuraciones como corrida hija (nested=True)
        cv_df = pd.DataFrame(search.cv_results_)
        for idx, row in cv_df.iterrows():
            with mlflow.start_run(run_name=f"{nombre_padre}_cfg_{idx+1:02d}", nested=True):
                params_limpios = {k.replace("modelo__", ""): str(v) for k, v in row["params"].items()}
                mlflow.log_params({
                    "tabla_origen": "obt_riesgo_abandono.csv",
                    "fecha_corte": FECHA_CORTE,
                    "semilla": SEMILLA,
                    "hash_dataset": HASH_DATOS,
                    **params_limpios,
                })
                mlflow.log_metrics({
                    "pr_auc_train": round(float(row["mean_train_score"]), 5),
                    "pr_auc_cv_mean": round(float(row["mean_test_score"]), 5),
                    "pr_auc_cv_std": round(float(row["std_test_score"]), 5),
                    "segundos_entrenamiento": round(float(row["mean_fit_time"]), 4),
                })
                mlflow.set_tags({
                    "fase": "tuning_child",
                    "familia": familia,
                    "candidato": "no",
                    "git_commit": GIT_COMMIT,
                })

        # Evaluar el mejor estimador encontrado en train y test
        best_pipe = search.best_estimator_
        prob_train = best_pipe.predict_proba(X_train)[:, 1]
        prob_test = best_pipe.predict_proba(X_test)[:, 1]
        pred_test = (prob_test >= UMBRAL_NEGOCIO).astype(int)

        pr_train = float(average_precision_score(y_train, prob_train))
        pr_cv_mean = float(search.best_score_)
        pr_cv_std = float(cv_df.loc[search.best_index_, "std_test_score"])
        pr_test = float(average_precision_score(y_test, prob_test))

        best_params_limpios = {k.replace("modelo__", ""): v for k, v in search.best_params_.items()}
        mlflow.log_params({
            "tabla_origen": "obt_riesgo_abandono.csv",
            "fecha_corte": FECHA_CORTE,
            "semilla": SEMILLA,
            "hash_dataset": HASH_DATOS,
            "n_configs_evaluadas": N_CONFIGS,
            **{k: str(v) for k, v in best_params_limpios.items()},
        })
        mlflow.log_metrics({
            "pr_auc_train": round(pr_train, 5),
            "pr_auc_cv_mean": round(pr_cv_mean, 5),
            "pr_auc_cv_std": round(pr_cv_std, 5),
            "pr_auc_test": round(pr_test, 5),
            "brecha_train_test": round(pr_train - pr_test, 5),
            "roc_auc_test": round(float(roc_auc_score(y_test, prob_test)), 5),
            "f1_test_060": round(float(f1_score(y_test, pred_test)), 5),
            "recall_test_060": round(float(recall_score(y_test, pred_test)), 5),
            "precision_test_060": round(float(precision_score(y_test, pred_test)), 5),
            "segundos_entrenamiento": round(seg_padre, 4),
        })
        mlflow.set_tags({
            "fase": "tuning",
            "familia": familia,
            "candidato": "no",
            "git_commit": GIT_COMMIT,
        })

        input_example = X_train.iloc[:5].copy()
        signature = infer_signature(input_example, best_pipe.predict_proba(input_example))
        mlflow.sklearn.log_model(
            sk_model=best_pipe,
            artifact_path="model",
            signature=signature,
            input_example=input_example,
        )

        mejores_tuneados.append({
            "run_id": parent_run.info.run_id,
            "nombre": nombre_padre,
            "familia": familia,
            "best_params": best_params_limpios,
            "estimator": best_pipe,
            "pr_auc_cv_mean": pr_cv_mean,
            "pr_auc_cv_std": pr_cv_std,
            "pr_auc_test": pr_test,
        })

df_tuning = pd.DataFrame(mejores_tuneados).drop(columns=["estimator"]).sort_values(
    by="pr_auc_cv_mean", ascending=False
).reset_index(drop=True)
display(df_tuning)

🏃 View run Tuning_HistGradientBoosting_cfg_01 at: http://127.0.0.1:5000/#/experiments/1/runs/3f1a03e4246a4cc9bfa4d15ccc838ebf
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
🏃 View run Tuning_HistGradientBoosting_cfg_02 at: http://127.0.0.1:5000/#/experiments/1/runs/93d53fd0f9634c03b65dbc2519c19e1b
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
🏃 View run Tuning_HistGradientBoosting_cfg_03 at: http://127.0.0.1:5000/#/experiments/1/runs/63a6341143a24e5182782b0c9fe7f419
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
🏃 View run Tuning_HistGradientBoosting_cfg_04 at: http://127.0.0.1:5000/#/experiments/1/runs/799a6a2c12d94a9cbae659393ab174cd
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
🏃 View run Tuning_HistGradientBoosting_cfg_05 at: http://127.0.0.1:5000/#/experiments/1/runs/570e6d4ed79a41c0a1b4a312d9f4865d
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
🏃 View run Tuning_HistGradientBoosting_cfg_06 at: http://127.0.0.1:500

🏃 View run Tuning_HistGradientBoosting at: http://127.0.0.1:5000/#/experiments/1/runs/a20017faeb7c4619b9b2a5e0119bb9a2
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
🏃 View run Tuning_MLPClassifier_cfg_01 at: http://127.0.0.1:5000/#/experiments/1/runs/9a60b46d69eb46848af43605a6bc403c
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
🏃 View run Tuning_MLPClassifier_cfg_02 at: http://127.0.0.1:5000/#/experiments/1/runs/ec1ec0ad8e15432aba9963b1eb613b60
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
🏃 View run Tuning_MLPClassifier_cfg_03 at: http://127.0.0.1:5000/#/experiments/1/runs/4ba768928da54a0688d8d0fd7803d260
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
🏃 View run Tuning_MLPClassifier_cfg_04 at: http://127.0.0.1:5000/#/experiments/1/runs/38c49e0fb8494d938f2052fa3848ae4a
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
🏃 View run Tuning_MLPClassifier_cfg_05 at: http://127.0.0.1:5000/#/experiments/1/runs/cbef2a860dc44470a94

🏃 View run Tuning_MLPClassifier at: http://127.0.0.1:5000/#/experiments/1/runs/672413a0bd0b489f9359f35ecf07722b
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
🏃 View run Tuning_LogisticRegression_cfg_01 at: http://127.0.0.1:5000/#/experiments/1/runs/6f3d7212dbbe44559da81d9ad4d874bf
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
🏃 View run Tuning_LogisticRegression_cfg_02 at: http://127.0.0.1:5000/#/experiments/1/runs/0d8eb335f27e4e70b1aa54458cb7c176
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
🏃 View run Tuning_LogisticRegression_cfg_03 at: http://127.0.0.1:5000/#/experiments/1/runs/e636b5cc45384ecba1cad1a04ad910a9
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
🏃 View run Tuning_LogisticRegression_cfg_04 at: http://127.0.0.1:5000/#/experiments/1/runs/f7743a4ce1704b7997313505cd011d40
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
🏃 View run Tuning_LogisticRegression_cfg_05 at: http://127.0.0.1:5000/#/experiments/1/runs/7

🏃 View run Tuning_LogisticRegression at: http://127.0.0.1:5000/#/experiments/1/runs/92446a9861994d22a0a7431b389f1eb1
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


,run_id,nombre,familia,best_params,pr_auc_cv_mean,pr_auc_cv_std,pr_auc_test
0,672413a0bd0b489f9359f35ecf07722b,Tuning_MLPClassifier,Red_Neuronal,"{'alpha': 0.003378449765619961, 'hidden_layer_...",0.983301,0.003423,0.982692
1,92446a9861994d22a0a7431b389f1eb1,Tuning_LogisticRegression,Lineal_Regularizado,"{'C': 73.92266140516048, 'class_weight': None}",0.982981,0.004608,0.982180
2,a20017faeb7c4619b9b2a5e0119bb9a2,Tuning_HistGradientBoosting,Boosting,"{'l2_regularization': 0.02528435468072164, 'le...",0.982542,0.004535,0.984011


Al evaluar `30` combinaciones aleatorias por cada una de las 3 mejores familias mediante `TimeSeriesSplit(n_splits=5)`, todos los modelos mejoraron su promedio en validación cruzada respecto a sus valores por defecto:
* **1.º `Tuning_MLPClassifier` (`Red_Neuronal`):** Sube de `0.98256` a **`0.98330 ± 0.00342`** en CV (`0.98269` en prueba) con regularización `alpha = 0.00338`, logrando el mejor rendimiento promedio y la menor desviación entre pliegues.
* **2.º `Tuning_LogisticRegression` (`Lineal_Regularizado`):** Sube de `0.98170` a **`0.98298 ± 0.00461`** en CV (`0.98218` en prueba) con `C = 73.92`, quedando en segundo lugar a apenas `0.00032` puntos del líder.
* **3.º `Tuning_HistGradientBoosting` (`Boosting`):** Sube de `0.98057` a **`0.98254 ± 0.00454`** en CV y obtiene la mayor métrica puntual en prueba (`0.98401`) con `l2_regularization = 0.0253`.

In [3]:
# Ordenar por desempeño en CV temporal (criterio sin fuga de test)
ranking = sorted(mejores_tuneados, key=lambda x: x["pr_auc_cv_mean"], reverse=True)
ganador = ranking[0]
segundo = ranking[1]

# Limpiar versiones previas del registro si existieran
try:
    for v in client.search_model_versions(f"name='{MODEL_REGISTRY_NAME}'"):
        client.delete_model_version(name=MODEL_REGISTRY_NAME, version=v.version)
except Exception:
    pass

# Registrar corrida final oficial marcada con candidato=final -> alias @champion
with mlflow.start_run(run_name=f"FINAL_{ganador['familia']}") as final_run:
    t0 = time.perf_counter()
    final_pipe = ganador["estimator"]
    final_pipe.fit(X_train, y_train)
    seg_final = time.perf_counter() - t0

    prob_train_f = final_pipe.predict_proba(X_train)[:, 1]
    prob_test_f = final_pipe.predict_proba(X_test)[:, 1]
    pred_test_f = (prob_test_f >= UMBRAL_NEGOCIO).astype(int)

    pr_train_f = float(average_precision_score(y_train, prob_train_f))
    pr_test_f = float(average_precision_score(y_test, prob_test_f))

    mlflow.log_params({
        "tabla_origen": "obt_riesgo_abandono.csv",
        "fecha_corte": FECHA_CORTE,
        "semilla": SEMILLA,
        "hash_dataset": HASH_DATOS,
        "umbral_negocio": UMBRAL_NEGOCIO,
        **{k: str(v) for k, v in ganador["best_params"].items()},
    })
    mlflow.log_metrics({
        "pr_auc_train": round(pr_train_f, 5),
        "pr_auc_cv_mean": round(ganador["pr_auc_cv_mean"], 5),
        "pr_auc_cv_std": round(ganador["pr_auc_cv_std"], 5),
        "pr_auc_test": round(pr_test_f, 5),
        "brecha_train_test": round(pr_train_f - pr_test_f, 5),
        "roc_auc_test": round(float(roc_auc_score(y_test, prob_test_f)), 5),
        "f1_test_060": round(float(f1_score(y_test, pred_test_f)), 5),
        "recall_test_060": round(float(recall_score(y_test, pred_test_f)), 5),
        "precision_test_060": round(float(precision_score(y_test, pred_test_f)), 5),
        "segundos_entrenamiento": round(seg_final, 4),
    })
    mlflow.set_tags({
        "fase": "final",
        "familia": ganador["familia"],
        "candidato": "final",
        "git_commit": GIT_COMMIT,
    })

    input_example = X_train.iloc[:5].copy()
    signature = infer_signature(input_example, final_pipe.predict_proba(input_example))
    mlflow.sklearn.log_model(
        sk_model=final_pipe,
        artifact_path="model",
        signature=signature,
        input_example=input_example,
        registered_model_name=MODEL_REGISTRY_NAME,
    )
    final_run_id = final_run.info.run_id

# Registrar el segundo mejor en el Model Registry como @challenger
mv_challenger = mlflow.register_model(
    model_uri=f"runs:/{segundo['run_id']}/model",
    name=MODEL_REGISTRY_NAME,
)

# Asignar alias champion (v1) y challenger (v2)
versiones = sorted(
    client.search_model_versions(f"name='{MODEL_REGISTRY_NAME}'"),
    key=lambda v: int(v.version),
)
v_champion = versiones[0].version
v_challenger = versiones[1].version

client.set_registered_model_alias(MODEL_REGISTRY_NAME, "champion", v_champion)
client.set_registered_model_alias(MODEL_REGISTRY_NAME, "challenger", v_challenger)

# Recuperar modelo desde models:/oralsolutions-modelo@champion y reproducir métrica exacta
modelo_recuperado = mlflow.sklearn.load_model(f"models:/{MODEL_REGISTRY_NAME}@champion")
prob_rec = modelo_recuperado.predict_proba(X_test)[:, 1]
pr_auc_recuperado = round(float(average_precision_score(y_test, prob_rec)), 5)
pr_auc_registrado = round(pr_test_f, 5)

assert pr_auc_recuperado == pr_auc_registrado, (
    f"Discrepancia al recuperar champion: {pr_auc_recuperado} vs {pr_auc_registrado}"
)

print("=== VERIFICACIÓN MODEL REGISTRY ===")
print(f"Modelo registrado : {MODEL_REGISTRY_NAME}")
print(f"Champion (@champion)  : versión {v_champion} | Familia={ganador['familia']} | run_id={final_run_id}")
print(f"Challenger (@challenger): versión {v_challenger} | Familia={segundo['familia']} | run_id={segundo['run_id']}")
print(f"PR-AUC registrado : {pr_auc_registrado} | PR-AUC recuperado : {pr_auc_recuperado} [EXACTO]")

# Exportar todas las corridas limpias a modelado/corridas_proyecto.csv
df_corridas = mlflow.search_runs(experiment_ids=[exp_id])
ruta_corridas = Path("corridas_proyecto.csv")
df_corridas.to_csv(ruta_corridas, index=False)
print(f"[OK] Exportadas {len(df_corridas)} corridas limpias a: {ruta_corridas.resolve()}")

Registered model 'oralsolutions-modelo' already exists. Creating a new version of this model...
2026/11/18 08:08:36 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: oralsolutions-modelo, version 7
Created version '7' of model 'oralsolutions-modelo'.
Registered model 'oralsolutions-modelo' already exists. Creating a new version of this model...
2026/11/18 08:08:36 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: oralsolutions-modelo, version 8
Created version '8' of model 'oralsolutions-modelo'.


🏃 View run FINAL_Red_Neuronal at: http://127.0.0.1:5000/#/experiments/1/runs/169e8821e2c54761bf602b21f86aec93
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
=== VERIFICACIÓN MODEL REGISTRY ===
Modelo registrado : oralsolutions-modelo
Champion (@champion)  : versión 7 | Familia=Red_Neuronal | run_id=169e8821e2c54761bf602b21f86aec93
Challenger (@challenger): versión 8 | Familia=Lineal_Regularizado | run_id=92446a9861994d22a0a7431b389f1eb1
PR-AUC registrado : 0.98269 | PR-AUC recuperado : 0.98269 [EXACTO]
[OK] Exportadas 101 corridas limpias a: C:\Users\ANDRES\Desktop\data-science-end-to-end\psets\proyecto\modelado\corridas_proyecto.csv


Siguiendo el criterio sin fuga de usar únicamente el desempeño en validación cruzada (`pr_auc_cv_mean`) para elegir ganador, se registró `FINAL_Red_Neuronal` (`run_id = 169e8821e2c54761bf602b21f86aec93`, etiqueta `candidato = final`) en el Model Registry bajo el nombre **`oralsolutions-modelo`**, asignándole el alias **`@champion`** (versión 7) y dejando a `Tuning_LogisticRegression` (`run_id = 92446a9861994d22a0a7431b389f1eb1`) como **`@challenger`** (versión 8). Al descargar el pipeline completo desde `models:/oralsolutions-modelo@champion`, se reprodujo exactamente **`PR-AUC = 0.98269`** sobre `X_test`, y se exportaron las `101` corridas limpias a `modelado/corridas_proyecto.csv`.